# 02 — Extract MIDRC ZIPs and inspect DICOMs

After notebook 01 succeeds, MIDRC delivers each series as a single ZIP
under `data/raw/<class>/<case_id>/<study_uid>/<series_uid>.zip`.

This notebook:
1. Extracts every ZIP into `data/extracted/`, preserving the class folder.
2. Reads DICOM headers (no pixel data) from the extracted tree.
3. Saves per-file metadata (PatientID-bearing, **gitignored**).
4. Groups by `SeriesInstanceUID` -> `outputs/sample_series_summary.csv`.
5. Prints counts so we can sanity-check that labels match the metadata
   *before* we scale up.

The actual logic lives in `scripts/extract_midrc_zips.py` and
`scripts/inspect_dicoms.py`; this notebook just imports/calls them so you can
poke the resulting DataFrames interactively.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT     = Path("/Users/jason/HelperAI")
RAW_DIR       = REPO_ROOT / "data" / "raw"
EXTRACTED_DIR = REPO_ROOT / "data" / "extracted"
OUT_DIR       = REPO_ROOT / "outputs"
PER_FILE_CSV   = OUT_DIR / "dicom_file_metadata.csv"
PER_SERIES_CSV = OUT_DIR / "sample_series_summary.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "scripts"))
import extract_midrc_zips as extractor
import inspect_dicoms as inspector

print("raw       :", RAW_DIR)
print("extracted :", EXTRACTED_DIR)
print("outputs   :", OUT_DIR)

## 1. Extract all ZIPs under `data/raw/` into `data/extracted/`

Already-extracted directories are skipped. Pass `--force` from the CLI (or
delete `data/extracted/` by hand) to force a re-extract.

In [ ]:
extractor.main([
    "--input",  str(RAW_DIR),
    "--output", str(EXTRACTED_DIR),
])

## 2. Inspect extracted DICOM headers

This will:
- recursively read every file under `data/extracted/`,
- skip non-DICOM / unreadable files with a warning,
- write `outputs/dicom_file_metadata.csv` (per-file, gitignored),
- write `outputs/sample_series_summary.csv` (per-series, safe to commit),
- print a summary.

In [ ]:
inspector.main([
    "--input",  str(EXTRACTED_DIR),
    "--output", str(PER_SERIES_CSV),
])

## 3. Interactive look at the resulting DataFrames

Reload the CSVs we just wrote so we can slice them with pandas.

In [ ]:
import pandas as pd

per_file_df = pd.read_csv(PER_FILE_CSV) if PER_FILE_CSV.exists() else pd.DataFrame()
series_df   = pd.read_csv(PER_SERIES_CSV) if PER_SERIES_CSV.exists() else pd.DataFrame()

print("per-file rows :", len(per_file_df))
print("series rows   :", len(series_df))
series_df.head(20)

### Sanity-check: do the metadata labels match the intended class?

We pulled `chest_xray` using `modality in {CR, DX}` and `head_ct` using
`modality == CT`. If the per-series `Modality` column disagrees with the
folder name (`source_class`), that's a label-quality red flag worth
investigating before scaling up.

In [ ]:
if not series_df.empty:
    cross = (
        series_df.groupby(["source_class", "Modality"], dropna=False)
                 .size()
                 .rename("n_series")
                 .reset_index()
    )
    print(cross.to_string(index=False))
else:
    print("No series found yet. Run cells above.")

In [ ]:
if not series_df.empty:
    desc = (
        series_df.groupby(["source_class", "StudyDescription"], dropna=False)
                 .size()
                 .rename("n_series")
                 .reset_index()
                 .sort_values(["source_class", "n_series"], ascending=[True, False])
    )
    print(desc.to_string(index=False))

### Done.

If the modality cross-tab looks right (CR/DX under `chest_xray`, CT under
`head_ct`) and `outputs/sample_series_summary.csv` has one row per imaging
series, this phase is complete. Do **not** download more data yet — next we
decide whether to scale up to a slightly larger sample or to iterate on the
filters.